# Testing One Benchmark Against One Model

This notebook explains, step by step, what happens when this repository runs **one benchmark** against **one model**.

It is written for a reader who may be new to machine learning evaluation, Python codebases, and this project in particular. The goal is not just to name the files involved, but to explain what they are for and how data moves through them.

## Basic Concepts and the Concrete Classes Behind Them

A few terms appear repeatedly in the code. It helps to define them first, and to immediately connect each one to a real class in this repository.

### What is a model?

A **model** is the system that looks at an input image and produces a text answer. In this repository, examples include `llava-7b`, `qwen25-vl`, and `gemma`.

A model name such as `"llava-7b"` is just a short string used by the code to decide which Python class or factory function to use.

A concrete example is the class `models.llava.Llava`.


| Attribute | Meaning |
| --- | --- |
| `name` | Short internal name for the model wrapper. In `Llava.__init__`, this is set to `"llava"`. |
| `model_id` | Hugging Face model identifier, for example `"llava-hf/llava-1.5-7b-hf"`. |
| `max_new_tokens` | Maximum number of output tokens the model is allowed to generate. |
| `stream` | Whether output is streamed token by token. |
| `load_in_4bit` | Whether the model should try low-memory 4-bit loading. |
| `processor` | The multimodal processor that prepares text and image inputs. |
| `model` | The actual neural network object used for generation. |

A few natural questions arise about these attributes:

#### Is `max_new_tokens` set by the programmer of this repository, or does it come already set?

In this repository, `max_new_tokens` is set by the repository's own code.

More precisely:

- `benchmark_runner.py` computes a token budget from benchmark defaults,
- the model factory is then called with that value,
- later, just before running a specific benchmark, the runner may update `model.max_new_tokens` again to the benchmark-specific value.

So this setting does **not** simply arrive preconfigured from Hugging Face. The local runner actively chooses it.

#### What does it mean to stream token by token?

A language model does not usually generate the full answer in one instant. It generates it piece by piece.

A **token** is a small unit of text, often smaller than a word. Streaming token by token means the wrapper begins exposing the generated output as it is produced, instead of waiting for the full answer to finish first.

In `models.llava.Llava.predict()`, when `stream=True`, the code creates a `TextIteratorStreamer` and reads output incrementally from it. When `stream=False`, the code waits for `self.model.generate(...)` to finish and only then decodes the result.

#### What does it mean to try low-memory 4-bit loading?

Neural network weights are usually stored in numeric formats such as 16-bit or 32-bit floating point values. A **4-bit** format stores each weight using much less memory.

So low-memory 4-bit loading means: try to load the model in a compressed representation so it uses less GPU memory or RAM.

In `Llava.__init__`, this happens only if:

- `load_in_4bit=True`,
- CUDA is available,
- the `bitsandbytes` package is installed.

If those conditions are not met, the code falls back to normal loading.

#### What is the `processor`?

The `processor` is the object that prepares raw inputs for the model.

For a multimodal model such as LLaVA, that means it handles tasks like:

- converting text into token IDs,
- converting the image into the tensor format expected by the model,
- packaging both pieces together into the input structure used by `generate(...)`.

In this file, the processor is loaded with `AutoProcessor.from_pretrained(...)`, so it is a Hugging Face processor object appropriate for the selected model.

#### What type of object is the `model` attribute?

For `models.llava.Llava`, the `model` attribute is the actual Hugging Face neural network object returned by `LlavaForConditionalGeneration.from_pretrained(...)`.

So conceptually:

- the wrapper class is `models.llava.Llava`,
- inside it, `self.model` is an instance of `transformers.LlavaForConditionalGeneration`.

That inner object is the one that actually runs `generate(...)`.

### What is a dataset?

A **dataset** is the source of examples used by the benchmark. For an image classification task, it provides images and their correct labels.

A concrete example is the class `dataset.imagenet1k.ImageNet1k`.

| Attribute | Meaning |
| --- | --- |
| `name` | Dataset name. Here it is `"imagenet-1k"`. |
| `split` | Which dataset split is used, such as `"validation"`. |
| `streaming` | Whether rows are streamed instead of fully downloaded up front. |
| `ds` | The Hugging Face dataset object returned by `load_dataset(...)`. |
| `class_labels` | The full list of class names extracted from dataset features, if available. |
| `labels` | The current public label list used by the benchmark code. |
| `idx2synset` | Optional mapping from class index to WordNet synset ID. |

### What is a benchmark?

A **benchmark** is a standardized test for models. It defines:

- what data to use,
- what task the model should perform,
- how to score the model's answer.

In this notebook, the example benchmark is `imagenet1k`, a classification benchmark. That means the model sees an image and is expected to choose the correct label for it.

Concrete benchmark classes in this path include `BaseBenchmark`, `ClassificationBenchmark`, and `ImageNet1kBenchmark`.

| Attribute | Meaning |
| --- | --- |
| `BaseBenchmark.dataset` | The dataset object the benchmark will read rows from. |
| `BaseBenchmark.name` | The benchmark's name, such as `"imagenet1k"`. |
| `BaseBenchmark.MAX_PROMPT_LABELS` | Upper bound on how many labels are shown in a labeling prompt. |
| `ClassificationBenchmark.max_edge` | Largest image edge allowed after resizing. |
| `ImageNet1kBenchmark.benchmark_name` | Class-level short name used by the runner: `"imagenet1k"`. |
| `ImageNet1kBenchmark.default_max_new_tokens` | Default token budget for this benchmark: `16`. |

### What is a dataset row?

A **row** is one example from a dataset. For an image classification task, one row usually contains:

- an image,
- one or more correct labels,
- sometimes extra metadata.

In the ImageNet path, a row is expected to behave like a Python dictionary with keys such as `"image"` and `"label"`.

### What is a prompt?

A **prompt** is the text instruction sent to the model along with the image. For a labeling task, the prompt usually tells the model to choose exactly one label from a provided list.

A concrete example comes from `BaseBenchmark.make_prompt()`:

For a labeling task, the base benchmark builds a prompt of this form:

```text
USER: <image>
Return exactly ONE label from this list (one item only, no extra words):
golden retriever, tabby cat, fire truck, espresso maker
ASSISTANT:
```

### What is a registry?

A **registry** is a lookup table. In Python terms, it behaves like a dictionary that maps a short name to the code object that should be used for that name.

For example:

- a benchmark registry maps names like `"imagenet1k"` to benchmark classes,
- a model registry maps names like `"llava-7b"` to functions that build models.

This lets the runner accept short user-facing names instead of forcing the caller to manually import Python classes.

### What does it mean to validate a name?

To **validate** a name means to check whether the name is allowed and known to the program.

For example, if the caller asks for `"llava-7b"`, the code checks whether that string exists in the model registry. If it does not, the program raises an error instead of continuing with an invalid request.

### What does instantiate mean?

To **instantiate** something means to create a concrete object from a class or factory function.

For example, the repository may store a model factory in a registry, and later call it to create the actual model object that will run predictions.

### What is a report?

A **report** is the final structured result of a benchmark run. It contains per-sample results and aggregate statistics such as accuracy or runtime information.

## What a Row Looks Like in This Path

In the ImageNet classification path, a row is expected to behave like a Python dictionary.

At minimum, the code in `ImageNet1k.get_image_from_row()` and `ImageNet1k.get_labels_img()` expects fields like these:

- `row["image"]`: the raw image payload,
- `row["label"]`: an integer class index.

A realistic simplified row therefore looks like this:

```python
row = {
    "image": <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x375>,
    "label": 207,
}
```

The exact image object may vary. From the source code, `row["image"]` may be either:

- a `PIL.Image.Image` instance, or
- a dictionary containing image bytes or a file path.

So the corresponding Python type checks would usually look like this:

```python
type(row)
# dict

type(row["label"])
# int

type(row["image"])
# often PIL.JpegImagePlugin.JpegImageFile or another PIL.Image.Image subclass
```

After `get_image_from_row(row)` runs, the returned image should be a PIL RGB image:

```python
img = dataset.get_image_from_row(row)
type(img)
# PIL.Image.Image (or a subclass)
```

The label text for the row is not stored directly as a string in the row. Instead, the integer index in `row["label"]` is resolved through `dataset.class_labels[row["label"]]`.

In [ ]:
from pprint import pprint
import inspect

from benchmarks.classification.imagenet1k import ImageNet1kBenchmark
from dataset import ImageNet1k
from models.llava import Llava

print("Benchmark class attributes:")
print("  benchmark_name =", ImageNet1kBenchmark.benchmark_name)
print("  default_max_new_tokens =", ImageNet1kBenchmark.default_max_new_tokens)
print()

print("Dataset constructor signature:")
print(" ", inspect.signature(ImageNet1k.__init__))
print()

print("Model constructor signature:")
print(" ", inspect.signature(Llava.__init__))
print()

print("Key method signatures:")
print("  ImageNet1k.get_samples:", inspect.signature(ImageNet1k.get_samples))
print("  ImageNet1k.get_image_from_row:", inspect.signature(ImageNet1k.get_image_from_row))
print("  Llava.predict:", inspect.signature(Llava.predict))


## Live Example: Fetch One Real Row and Display Its Image

The next cell is meant to be run interactively in the notebook. It will try to fetch one real ImageNet validation row, show the Python types involved, print the label information, and display the image.

Important caveat: `ImageNet1k` is often gated on Hugging Face. If you do not have access configured locally, the cell may fail. That is expected and does not mean the notebook is wrong.

In [ ]:
from IPython.display import display

from dataset import ImageNet1k

ds = ImageNet1k(split="validation", streaming=True)
row = ds.get_samples(1)[0]

print("type(row):", type(row))
print("row keys:", list(row.keys()))
print("type(row['label']):", type(row["label"]))
print("type(row['image']):", type(row["image"]))
print("row['label']:", row["label"])
print("dataset.class_labels[row['label']]:", ds.class_labels[row["label"]])
print("valid labels for this row:", ds.get_labels_img(row)[:5])

img = ds.get_image_from_row(row)
print("type(img):", type(img))
print("img.mode:", img.mode)
print("img.size:", img.size)

display(img)


## Scope of This Notebook

To keep the explanation manageable, this notebook describes a simplified but representative case:

1. the caller asks for exactly one benchmark,
2. the caller asks for exactly one model,
3. the benchmark is a **labeling** benchmark, meaning the model must return a label such as `"golden retriever"` or `"tabby cat"`.

The entry point is `run_benchmark_matrix()` in `benchmark_runner.py`.

Even though the function name says "matrix", it still handles the one-benchmark, one-model case. In that simplified case, the matrix has just one cell: one model evaluated on one benchmark.

## The Main Question the Code Is Answering

The core question is:

> Given a model name, a benchmark name, and a number of samples, how does the repository produce a JSON file containing the model's results?

At a high level, the answer is:

1. find the requested model and benchmark by name,
2. create the corresponding Python objects,
3. ask the benchmark to run the model on dataset samples,
4. collect scores and telemetry,
5. write the results to disk.

## High-Level Flow, Explained Slowly

For one benchmark name and one model name, the execution chain is roughly this:

1. `benchmark_runner.py` checks whether the requested benchmark name and model name are recognized.
2. It creates the model object corresponding to the requested model name.
3. It creates the benchmark object corresponding to the requested benchmark name.
4. The benchmark prepares dataset examples and the list of labels that may appear in prompts.
5. For each sample image, the benchmark builds a prompt and calls `model.predict(image, prompt)`.
6. The benchmark compares the model's output with the correct label or labels.
7. The benchmark stores per-sample results and aggregate statistics.
8. The runner writes the final report to a JSON file.

If you want one sentence to remember, it is this: **the runner chooses what to run, and the benchmark performs the actual evaluation loop**.

## `benchmark_runner.py`: The Top-Level Coordinator

This file is the main entry point for matrix-style execution. Its job is not to do the scoring logic itself. Its job is to coordinate the pieces.

A useful mental model is:

- `benchmark_runner.py` is the dispatcher,
- benchmark classes know how to evaluate a task,
- model classes know how to generate answers,
- dataset classes know how to provide images and labels.

### Important global structures

| Name | What it stores | Why it matters |
| --- | --- | --- |
| `BENCHMARKS` | A collection of benchmark factories or classes. | This is the source from which benchmark names are derived. |
| `MODELS` | A collection of `ModelSpec` entries. Each one stores a short model name and a factory function. | This is how the code knows what `"llava-7b"` or `"gemma"` means. |
| `BENCHMARK_TOKENS` | A mapping from benchmark name to a default token budget. | Different benchmarks may need different output lengths from the model. |

### Important helper functions

| Function | Plain-English role |
| --- | --- |
| `_resolve_benchmark_registry(...)` | Converts the benchmark source into a dictionary-like mapping from benchmark name to benchmark factory. |
| `_resolve_model_registry(...)` | Converts the model source into a dictionary-like mapping from model name to model factory. |
| `get_benchmark_token_defaults(...)` | Builds the per-benchmark default token limits. |
| `_write_report(output_dir, model_name, benchmark_name, report)` | Writes the final JSON payload to disk. |
| `run_benchmark_matrix(model_names, benchmark_names, num_samples, ...)` | Coordinates the whole process. |

### What are `model_names` and `benchmark_names`?

They are lists of strings provided by the caller. For example:

- `model_names = ["llava-7b"]`
- `benchmark_names = ["imagenet1k"]`

Even in the one-model, one-benchmark case, the function still accepts lists because the same function can also run multiple combinations.

### What `run_benchmark_matrix()` does in the one-model, one-benchmark case

1. It resolves the benchmark registry and model registry into lookup tables.
2. If no custom token settings were passed in, it creates default token settings from the benchmark definitions.
3. It checks `num_samples`. If `num_samples < 1`, it raises an error because running zero samples would make no sense.
4. It checks whether every requested model name is present in the model registry.
5. It checks whether every requested benchmark name is present in the benchmark registry.
6. It computes an initial token budget. In the one-benchmark case, this is effectively the token budget for that single benchmark.
7. It creates the model object by calling the model factory associated with the chosen model name.
8. It measures how long model creation took and stores that as `model_load_time_seconds`.
9. It chooses the benchmark factory for the requested benchmark name.
10. It updates `model.max_new_tokens` if the model object exposes that setting.
11. It instantiates the benchmark with `streaming=True`.
12. It calls `benchmark.run(...)`, which performs the real evaluation loop.
13. It inserts model load time into `report["stats"]`.
14. It writes the report JSON to `results/<model>_<benchmark>.json`.
15. It appends a short summary dictionary to the return value.

### Why does this file not directly score predictions?

Because scoring depends on the task. Classification, detection, captioning, and visual question answering all need different logic. The runner stays general and lets each benchmark class handle task-specific details.

## `benchmarks/__init__.py`: Lazy Access to Benchmark Classes

This file supports **lazy imports**. That means the code can refer to benchmark class names without importing every benchmark module immediately at startup.

The important function is `__getattr__(name)`.

In plain English, when Python asks the `benchmarks` package for something like `ImageNet1kBenchmark`, `__getattr__()` decides which module actually contains that class and loads it.

Why this matters:

- it keeps the package interface clean,
- it avoids loading everything up front,
- it lets `benchmark_runner.py` import many benchmark names from one place.

## `benchmarks/datas/benchmark_wrappers.py`: Connecting a Benchmark Name to a Concrete Dataset

This file acts as a bridge between a **generic benchmark type** and a **specific dataset**.

For the example path:

- `ImageNet1kBenchmark(...)` creates an `ImageNet1k(...)` dataset object,
- then it passes that dataset into `ClassificationBenchmark`.

This is important because `ClassificationBenchmark` contains general logic for classification tasks, while `ImageNet1k` contains dataset-specific logic such as how to load rows and which labels are valid.

A beginner-friendly way to think about it is:

- `ClassificationBenchmark` says **how** to run a labeling test,
- `ImageNet1k` says **which data** the test uses.

## `benchmarks/classification.py`: The Benchmark Type for Labeling Tasks

This file defines the benchmark subclass used for classification or labeling tasks.

| Function | Plain-English role |
| --- | --- |
| `ClassificationBenchmark.__init__(dataset, name)` | Creates a classification benchmark and passes the dataset and benchmark name up to the base benchmark class. |
| `get_image_for_row(row)` | Pulls the image out of a dataset row and prepares it for the model. |
| `_resize_image(image)` | Resizes the image so it is not too large. |

This subclass is intentionally small. It mainly defines how to get images for classification rows. Much of the shared benchmark logic stays in `BaseBenchmark`.

That design is useful because many classification datasets can share the same evaluation loop.

## `benchmarks/base.py`: The Core Evaluation Loop

This is the most important file for understanding what happens during a benchmark run.

The base benchmark class contains the common loop that:

- selects rows,
- prepares labels,
- builds prompts,
- calls the model,
- scores the result,
- accumulates statistics.

### Important functions in the labeling path

| Function | Plain-English role |
| --- | --- |
| `BaseBenchmark.__init__(dataset, name)` | Stores the dataset object and the benchmark name. |
| `get_candidate_labels(rows)` | Asks the dataset for the overall pool of labels that may be shown in prompts. |
| `get_valid_labels_for_row(row)` | Asks the dataset which answers should count as correct for one specific row. |
| `get_prompt_labels_for_row(row, labels)` | Chooses which labels to include in the prompt for this row. |
| `make_rng_for_row(row)` | Creates a deterministic random generator so prompt-label sampling is reproducible. |
| `prepare(n, label_sample_size)` | Collects rows and candidate labels before the main loop starts. |
| `make_prompt(labels, row, image)` | Builds the instruction text sent to the model. |
| `evaluate_prediction(row, prediction, image)` | Decides whether the model's answer should count as correct. |
| `analyze_prediction(...)` | Computes extra behavior statistics beyond simple correctness. |
| `build_run_statistics(sample_results, wall_clock_seconds)` | Aggregates per-sample results into run-level metrics. |
| `run(model, n, label_sample_size, show_progress, on_sample)` | Executes the full benchmark loop. |

### What `prepare()` does

1. It asks the dataset for enough rows by calling `dataset.get_samples(max(n, label_sample_size))`.
2. It takes the first `n` rows as the actual evaluation rows.
3. It asks the dataset to build the candidate label pool from the selected rows for label gathering.
4. It returns `(rows, labels)`.

Why are `rows` and `labels` separate?

- `rows` are the actual examples being scored.
- `labels` are the answer choices that may appear in prompts.

### What `run()` does for each sample

For each dataset row, the benchmark performs a sequence of small steps:

1. Load the image for the row with `get_image_for_row(row)`.
2. Decide which labels to show in the prompt with `get_prompt_labels_for_row(row, labels)`.
3. Build the textual instruction with `make_prompt(...)`.
4. Start a `ResourceSampler` so the code can measure RAM, GPU memory, and related telemetry during prediction.
5. Call `model.predict(image, prompt)`.
6. Stop the resource sampler.
7. Optionally parse predicted boxes, even though the normal labeling path usually has no boxes.
8. Evaluate whether the prediction is correct.
9. Retrieve ground-truth boxes for the row; for ordinary labeling tasks this is usually an empty list.
10. Run extra analysis such as hallucination-related statistics.
11. If supported, count the number of output tokens generated by the model.
12. Store a result dictionary for that sample.

After all rows are processed, `run()` aggregates the sample results into final summary statistics and returns the report.

### What does `evaluate_prediction()` really mean here?

The model may output text in a slightly messy form, such as extra punctuation or capitalization differences. The benchmark therefore normalizes the text before comparing it to the correct labels.

For example, `"Golden Retriever"` and `"golden retriever"` should usually be treated as the same label after normalization.

## `benchmarks/resource_sampler.py`: Measuring Resource Usage

This file tracks how many computational resources are used while the model is generating an answer.

| Function | Plain-English role |
| --- | --- |
| `ResourceSampler.__init__(interval_seconds)` | Prepares the sampler and its internal state. |
| `start()` | Begins measuring resource usage. |
| `stop()` | Stops measuring and returns the collected metrics. |
| `_run()` | The background loop that repeatedly samples resource usage. |

Why this exists:

- performance is not only about accuracy,
- the repository also wants telemetry such as memory usage and runtime,
- that information can help compare models beyond just right versus wrong answers.

## `benchmarks/datas/__init__.py`: Lazy Access to Dataset Classes

This file plays the same role for datasets that `benchmarks/__init__.py` plays for benchmarks.

Its `__getattr__(name)` function resolves dataset class names such as `ImageNet1k` to the module where the class is actually defined.

For the running example:

- `ImageNet1k` -> `benchmarks.datas.class_imagenet1k`

This indirection keeps imports organized and lets the caller refer to dataset classes through the package interface.

## `benchmarks/datas/base.py`: The Dataset Interface

This file defines the methods a dataset must provide so that `BaseBenchmark` can use it.

A useful way to think about it is: the benchmark loop needs certain capabilities from any dataset, and `BaseDataset` documents those capabilities.

| Function | Plain-English role |
| --- | --- |
| `BaseDataset.__iter__()` | Defines how to iterate through dataset rows. |
| `get_labels(rows)` | Builds the candidate label pool from rows. |
| `get_samples(n)` | Selects or returns `n` rows. |
| `get_image_from_row(row)` | Extracts the image object from a row. |
| `normalize_text(text)` | Normalizes text for matching. |
| `label_map_normalized()` | Builds normalized lookup tables for label matching. |
| `is_valid_label(label)` | Checks whether a label belongs to the dataset's label set. |

### Why text normalization matters

If the benchmark compared raw strings directly, trivial formatting differences could incorrectly count as mistakes. Normalization reduces that problem by lowercasing text and removing punctuation or other formatting differences before matching.

## `benchmarks/datas/class_imagenet1k.py`: A Concrete Dataset Implementation

This file is the dataset-specific implementation for the ImageNet-1k benchmark path.

| Function | Plain-English role |
| --- | --- |
| `ImageNet1k.__init__(split, streaming, dataset_id, use_auth_token)` | Configures and loads access to the dataset, including class label information. |
| `__iter__()` | Iterates over dataset rows. |
| `get_samples(n)` | Retrieves the first `n` rows needed for the benchmark. |
| `get_image_from_row(row)` | Converts the row's stored image into a usable RGB PIL image. |
| `get_labels_img(row)` | Returns the labels that should count as correct for this row. |
| `get_labels(rows)` | Combines labels across rows to build the candidate prompt-label pool. |

### The difference between `get_labels_img(row)` and `get_labels(rows)`

These names are easy to confuse, but they serve different purposes:

- `get_labels_img(row)` answers: **what labels are correct for this one sample?**
- `get_labels(rows)` answers: **what labels should be available as answer choices across these samples?**

That distinction matters because the model may be shown a larger pool of candidate labels in the prompt, but only a smaller subset is correct for the current image.

### Helper functions

- `_synset_from_imagenet_synset_id(synset_id)`
- `_ancestor_synsets(syn, max_depth)`

These helpers matter when the benchmark wants to treat certain parent classes as acceptable answers, not just the most specific label.

## `models/__init__.py`: Lazy Access to Model Classes

This file provides the same lazy-import pattern for models.

When the code refers to names such as `Llava`, `Gemma`, `Qwen25VL`, or `SmallLlava`, the package can resolve those names to the correct modules without importing every model file immediately.

## `models/base.py`: What the Benchmark Expects from Any Model

The benchmark loop does not need to know every detail of every model implementation. It only needs a small common interface.

| Function | Plain-English role |
| --- | --- |
| `BaseModel.predict(image, prompt)` | Given an image and a prompt, produce the model's text output. |
| `get_tokenizer()` | Return or locate the tokenizer associated with the model. |
| `count_text_tokens(text)` | Count how many text tokens appear in generated output. |

The single most important method is `predict(image, prompt)`. As long as a model wrapper can do that, the benchmark can use it.

## `models/llava.py`: One Concrete Model Wrapper

This file is one example of a model wrapper. Other model files such as `models/gemma.py`, `models/qwen25_vl.py`, and `models/small_llava.py` serve the same broad purpose.

| Function | Plain-English role |
| --- | --- |
| `Llava.__init__(model_id, max_new_tokens, stream, load_in_4bit)` | Loads the model and processor, and stores generation settings. |
| `_load_with_local_fallback(loader, model_id, artifact_name, ...)` | Tries to load normally, then tries a local fallback if needed. |
| `_synchronize_processor_with_model_config()` | Repairs processor settings needed for correct image-token handling. |
| `predict(image, prompt)` | Converts inputs into the model's expected format, runs generation, decodes the output, and returns text. |
| `_prepare_prompt(prompt)` | Converts the benchmark prompt into the chat-style format expected by the model. |
| `_extract_instruction(prompt)` | Removes wrapper tokens and isolates the instruction text. |
| `_repair_image_token_mismatch(prompt, image, inputs)` | Handles mismatches between expected and actual image-token counts. |
| `_clean_output_text(output_text, prompt)` | Cleans the generated text before it is evaluated. |
| `_normalize_for_matching(text)` | Normalizes text so labels can be matched more reliably. |

The key idea is that benchmark code should not need to know how each model internally formats prompts or handles image tokens. That complexity stays inside the model wrapper.

## Exact Call Chain for the Example Path

Under the assumptions of this notebook, the call chain is:

1. `benchmark_runner.run_benchmark_matrix(["llava-7b"], ["imagenet1k"], num_samples)` is called.
2. The runner resolves the model registry and benchmark registry into name-to-object mappings.
3. The runner verifies that `"llava-7b"` is a known model name.
4. The runner verifies that `"imagenet1k"` is a known benchmark name.
5. The runner creates the `Llava` model object through the model factory stored for `"llava-7b"`.
6. The runner finds the benchmark factory for `"imagenet1k"`.
7. That benchmark factory constructs `ImageNet1kBenchmark`.
8. `ImageNet1kBenchmark.__init__()` constructs an `ImageNet1k` dataset object.
9. The classification benchmark is initialized on top of that dataset.
10. `benchmark.run(model, n, label_sample_size, show_progress=False)` begins the evaluation.
11. `BaseBenchmark.prepare(...)` gets rows and candidate labels from the dataset.
12. For each row, the benchmark loads the image, builds the prompt, and calls `Llava.predict(image, prompt)`.
13. The benchmark evaluates the prediction and stores the per-sample result.
14. After all rows are processed, the benchmark builds run-level statistics.
15. The runner inserts model-load timing into the report.
16. The runner writes the JSON file by calling `_write_report(...)`.

## Sample-Level Data Flow: What Happens to One Image

To make the process concrete, imagine one dataset row representing one image.

The data for that row moves through the system like this:

1. The row is selected by `get_samples()`.
2. `get_image_from_row()` extracts the image object from the row.
3. `get_labels_img(row)` provides the labels that should count as correct for this image.
4. `get_labels(rows)` contributes to the larger candidate label pool used for prompt construction.
5. `get_prompt_labels_for_row(...)` chooses which labels to show the model for this particular sample.
6. `make_prompt()` turns those labels into a textual instruction.
7. `predict()` returns the model's raw text output.
8. `evaluate_prediction()` normalizes the output and compares it to the valid labels.
9. The benchmark stores whether the answer was correct, what the prediction text was, which labels were valid, and what telemetry was recorded.

This is the core loop repeated for every sample.

## What the Output JSON Looks Like

The JSON written by `_write_report()` has this shape:

```json
{
  "model": "...",
  "benchmark": "...",
  "report": {
    "benchmark": "...",
    "dataset": "...",
    "num_samples": ...,
    "num_candidate_labels": ...,
    "results": [...],
    "stats": {...}
  }
}
```

### What the top-level fields mean

- `model`: the short model name used for the run.
- `benchmark`: the benchmark name used for the run.
- `report`: the detailed payload returned by the benchmark.

### What the `report` section generally contains

- `benchmark`: the benchmark's own name.
- `dataset`: the dataset involved.
- `num_samples`: how many examples were evaluated.
- `num_candidate_labels`: how many labels were available in the candidate pool.
- `results`: a list of per-sample result dictionaries.
- `stats`: aggregated statistics for the full run.

In the one-model, one-benchmark case, `run_benchmark_matrix()` also returns a one-element summary list containing:

- the model name,
- the benchmark name,
- the number of samples,
- the chosen `max_new_tokens`,
- the path to the saved results file.

## What Is Not Part of This Specific Path

This notebook is intentionally about the `benchmark_runner.py` execution path for a classification benchmark.

The following are related to the repository, but are not the core path described here:

- `main.py`, which is a separate manual runner with UI hooks,
- scripts under `comparison/`, which analyze saved results after benchmark execution,
- non-classification benchmark files such as `benchmarks/detection.py`, `benchmarks/captioning.py`, and `benchmarks/visualqa.py`.

This matters because large repositories often contain multiple entry points. A newcomer can easily get lost by mixing them together.

## Summary

If you remember only the main idea, remember this sequence:

1. `benchmark_runner.py` receives short names such as `"llava-7b"` and `"imagenet1k"`.
2. It checks that those names are valid by looking them up in registries.
3. It creates the corresponding model object and benchmark object.
4. The benchmark uses a dataset to fetch rows, build prompts, call the model, and score predictions.
5. The benchmark returns a structured report.
6. The runner saves that report as JSON.

So the repository is organized around a separation of responsibilities:

- the runner coordinates,
- the benchmark evaluates,
- the dataset supplies data,
- the model generates answers,
- the report records the outcome.